# DeepTopic motif hit analysis

Analyze MoDISco seqlet hits across topics — the CREsted analog of ChromBPNet finemo hit calling.
Each MoDISco seqlet IS an attribution-aware motif hit: a region where the model's contribution
scores match a known motif pattern. Unlike FIMO (sequence-only), these hits are model-specific
and reflect each topic's attribution profile.

**Key difference from ChromBPNet:** each topic has its own attribution profile, so hits are
per-topic by construction.

**Approach:**
1. Extract seqlet-level data from MoDISco h5 files (example_idx, contrib_scores, pattern identity)
2. Map raw patterns to clustered motifs via cluster_key.txt
3. Build per-topic hit tables: peak x motif binary/count/importance
4. Aggregate across topics for hit count heatmaps and importance analysis

Adapted from `bin/3_single_task_models/5_motif_hit_analysis.ipynb` (ChromBPNet/finemo).

**Inputs:**
- `crested_model/modisco/Topic*/Topic*.modisco.h5` — MoDISco results (seqlet-level hits)
- `crested_model/motifs/cluster/cluster_key.txt` — pattern → cluster mapping
- `binarized/beds/Topic*.bed` — peak coordinates per topic

**Outputs:**
- `crested_model/motifs/hits/hit_count_matrix.csv` — motif x topic hit counts
- `crested_model/motifs/hits/hit_importance_matrix.csv` — motif x topic importance
- `crested_model/motifs/hits/hit_count_clustermap.pdf`
- `crested_model/motifs/hits/hit_importance_clustermap.pdf`
- `crested_model/motifs/hits/per_topic/Topic*.hits.tsv` — seqlet-level hit tables

# Set-up

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd

import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
# Paths
dataset = 'sc-islet-differentiation_10X-Multiome'
subset = 'endocrine_replicate'
base_dir = f'/cellar/users/aklie/data/datasets/{dataset}'
results_dir = f'{base_dir}/results/4_topic_models/{subset}'
motif_dir = f'{results_dir}/crested_model/motifs'
modisco_dir = f'{results_dir}/crested_model/modisco'
bed_dir = f'{results_dir}/binarized/beds'

path_cluster_key = f'{motif_dir}/cluster/cluster_key.txt'
path_motif_annotation = f'{motif_dir}/tfs_initial.txt'
path_topic_cats = f'{results_dir}/summary/topic_categories.tsv'

path_out = f'{motif_dir}/hits'
per_topic_dir = f'{path_out}/per_topic'
os.makedirs(per_topic_dir, exist_ok=True)

all_topics = [f'Topic{i}' for i in range(1, 51)]

In [ ]:
# Load cluster key: raw pattern -> cluster name
pattern_to_cluster = {}
with open(path_cluster_key) as f:
    for line in f:
        parts = line.strip().split('\t')
        cluster = parts[0]
        for member in parts[1].split(','):
            pattern_to_cluster[member] = cluster

# Load annotations
initial_motifs = pd.read_csv(path_motif_annotation, sep='\t', header=None, skiprows=0,
                              names=['cluster_name', 'annotation', 'evalue'])
annotation_dict = initial_motifs.set_index('cluster_name')['annotation'].to_dict()
annot_short = {k: v.split('_')[0] for k, v in annotation_dict.items()}
all_clusters = initial_motifs['cluster_name'].tolist()

# Topic categories
topic_cats = pd.read_csv(path_topic_cats, sep='\t')
cat_colors = {
    'one-to-one': '#2ca02c', 'lineage': '#1f77b4', 'shared': '#ff7f0e',
    'structural': '#7f7f7f', 'batch-driven': '#d62728', 'unannotated': '#bcbd22',
}
topic_cat_dict = dict(zip(topic_cats['topic'], topic_cats['category']))

print(f'{len(pattern_to_cluster)} raw patterns -> {len(all_clusters)} clusters')

# Extract seqlet hits from MoDISco h5 files

For each topic, extract every seqlet's peak index, position, pattern identity,
and contribution score importance. Map to clustered motif names.

In [ ]:
# Extract all seqlet hits per topic
count_matrix = pd.DataFrame(0, index=all_clusters, columns=all_topics, dtype=int)
importance_matrix = pd.DataFrame(0.0, index=all_clusters, columns=all_topics)
peak_motif_data = {}  # topic -> list of (peak_idx, cluster, importance)

for topic in all_topics:
    h5_path = f'{modisco_dir}/{topic}/{topic}.modisco.h5'
    if not os.path.exists(h5_path):
        print(f'  Missing: {topic}')
        continue

    hits = []
    with h5py.File(h5_path, 'r') as f:
        for group_name in ['pos_patterns', 'neg_patterns']:
            if group_name not in f:
                continue
            for pattern_name in f[group_name]:
                raw_name = f'{topic}.{pattern_name}.pfm'
                cluster = pattern_to_cluster.get(raw_name)
                if not cluster or cluster not in count_matrix.index:
                    continue

                seqlets = f[f'{group_name}/{pattern_name}/seqlets']
                example_idx = seqlets['example_idx'][:]
                start = seqlets['start'][:]
                end = seqlets['end'][:]
                contrib = seqlets['contrib_scores'][:]

                # Per-seqlet importance: sum of absolute contribution scores
                importance = np.abs(contrib).sum(axis=(1, 2))

                count_matrix.loc[cluster, topic] += len(example_idx)
                importance_matrix.loc[cluster, topic] += importance.sum()

                for k in range(len(example_idx)):
                    hits.append({
                        'peak_idx': int(example_idx[k]),
                        'start': int(start[k]),
                        'end': int(end[k]),
                        'pattern': raw_name,
                        'cluster': cluster,
                        'importance': float(importance[k]),
                        'group': group_name,
                    })

    # Save per-topic hit table
    if hits:
        hit_df = pd.DataFrame(hits)
        hit_df.to_csv(f'{per_topic_dir}/{topic}.hits.tsv', sep='\t', index=False)
        n_peaks = hit_df['peak_idx'].nunique()
        n_motifs = hit_df['cluster'].nunique()
        print(f'  {topic}: {len(hits):,} seqlets, {n_peaks:,} peaks, {n_motifs} motifs')
    peak_motif_data[topic] = hits

print(f'\nTotal seqlets: {count_matrix.sum().sum():,}')

# Hit count and importance heatmaps

In [ ]:
# Filter to motifs with hits in at least 3 topics
active = count_matrix.index[count_matrix.gt(0).sum(axis=1) >= 3]
print(f'Active motifs (>= 3 topics): {len(active)} / {len(all_clusters)}')

# Z-score across topics
count_z = count_matrix.loc[active].apply(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0, axis=1
).fillna(0)

# Topic column colors
col_colors = [cat_colors.get(topic_cat_dict.get(t, ''), '#999999') for t in all_topics]

# Labels
row_labels = [f'{annot_short.get(m, m)}  ({m})' for m in active]

In [ ]:
# Hit count clustermap
plot_count = count_z.copy()
plot_count.index = row_labels

g = sns.clustermap(
    plot_count,
    cmap='coolwarm',
    center=0,
    figsize=(16, max(12, len(active) * 0.35)),
    row_cluster=True,
    col_cluster=True,
    col_colors=col_colors,
    cbar_kws={'label': 'Z-scored seqlet count'},
    yticklabels=True,
    xticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=7, rotation=45, ha='right')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=cat) for cat, c in cat_colors.items()]
g.ax_heatmap.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.15, 1.0),
                     title='Topic category', fontsize=7, title_fontsize=8)

plt.savefig(f'{path_out}/hit_count_clustermap.pdf', dpi=300, bbox_inches='tight')
print('Saved hit count clustermap')
plt.show()

In [ ]:
# Importance-weighted clustermap
imp_z = importance_matrix.loc[active].apply(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0, axis=1
).fillna(0)
plot_imp = imp_z.copy()
plot_imp.index = row_labels

g = sns.clustermap(
    plot_imp,
    cmap='coolwarm',
    center=0,
    figsize=(16, max(12, len(active) * 0.35)),
    row_cluster=True,
    col_cluster=True,
    col_colors=col_colors,
    cbar_kws={'label': 'Z-scored total importance'},
    yticklabels=True,
    xticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=7, rotation=45, ha='right')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)

g.ax_heatmap.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.15, 1.0),
                     title='Topic category', fontsize=7, title_fontsize=8)

plt.savefig(f'{path_out}/hit_importance_clustermap.pdf', dpi=300, bbox_inches='tight')
print('Saved importance clustermap')
plt.show()

# Hit quality and summary statistics

In [ ]:
# Per-topic QC
qc_rows = []
for topic in all_topics:
    hits_path = f'{per_topic_dir}/{topic}.hits.tsv'
    if not os.path.exists(hits_path):
        continue
    df = pd.read_csv(hits_path, sep='\t')
    qc_rows.append({
        'topic': topic,
        'n_hits': len(df),
        'n_peaks_with_hits': df['peak_idx'].nunique(),
        'n_motifs': df['cluster'].nunique(),
        'median_importance': df['importance'].median(),
        'mean_importance': df['importance'].mean(),
        'hits_per_peak': len(df) / df['peak_idx'].nunique(),
    })

qc = pd.DataFrame(qc_rows).set_index('topic')
qc.to_csv(f'{path_out}/hit_quality_summary.csv')

print(f'Median hits per topic: {qc["n_hits"].median():,.0f}')
print(f'Median hits per peak: {qc["hits_per_peak"].median():.1f}')
print(f'Median motifs per topic: {qc["n_motifs"].median():.0f}')

In [ ]:
# Quality distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(qc['n_hits'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Total seqlet hits')
axes[0].set_ylabel('Number of topics')
axes[0].set_title('Hits per topic')

axes[1].hist(qc['hits_per_peak'], bins=20, color='coral', edgecolor='white')
axes[1].set_xlabel('Hits per peak')
axes[1].set_title('Hit density per topic')

axes[2].hist(qc['median_importance'], bins=20, color='mediumpurple', edgecolor='white')
axes[2].set_xlabel('Median seqlet importance')
axes[2].set_title('Hit quality per topic')

plt.tight_layout()
plt.savefig(f'{path_out}/hit_quality_distributions.pdf', dpi=150, bbox_inches='tight')
plt.show()

# Motif composition barplot

In [ ]:
# Top motifs by total importance
top_n = 15
total_imp = importance_matrix.sum(axis=1).sort_values(ascending=False)
top_motif_names = total_imp.head(top_n).index.tolist()

# Importance fractions
imp_frac = importance_matrix.div(importance_matrix.sum(axis=0), axis=1).fillna(0)
plot_df = imp_frac.loc[top_motif_names].T.copy()
plot_df['Other'] = 1 - plot_df.sum(axis=1)

# Rename to annotations
plot_df.columns = [annot_short.get(m, m) if m != 'Other' else 'Other' for m in plot_df.columns]

fig, ax = plt.subplots(figsize=(16, 6))
plot_df.plot.bar(stacked=True, ax=ax, colormap='tab20', width=0.85)
ax.set_ylabel('Fraction of total importance')
ax.set_title('Motif composition by topic (seqlet importance)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{path_out}/motif_composition_barplot.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Save results

In [ ]:
count_matrix.to_csv(f'{path_out}/hit_count_matrix.csv')
importance_matrix.to_csv(f'{path_out}/hit_importance_matrix.csv')
count_z.to_csv(f'{path_out}/hit_count_zscore.csv')
imp_z.to_csv(f'{path_out}/hit_importance_zscore.csv')
imp_frac.to_csv(f'{path_out}/hit_importance_fraction.csv')
print(f'Saved all results to {path_out}/')

# DONE!

---